# 24 — Tensor Autograd and Gradient Checking

**Master syllabus coverage:** V2 3.5, 3.11

## Why this matters

Tensor derivatives introduce matrix orientation, reductions, and broadcasting. Gradient checks turn those implementation details into testable numerical contracts.

## Learning objectives

- Derive tensor gradients for an affine layer and mean-squared error.
- Reduce gradients correctly through broadcast operations.
- Implement elementwise central finite-difference checks.
- Compare manual, numerical, and PyTorch gradients in float64.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Tensor reverse mode is scalar reverse mode plus shape rules

For $Y=XW+b$ and scalar loss $L$, let upstream $G=\partial L/\partial Y$:

$$\frac{\partial L}{\partial X}=GW^\top,\qquad
\frac{\partial L}{\partial W}=X^\top G,\qquad
\frac{\partial L}{\partial b}=\sum_{\text{broadcast axes}}G.$$

The matrix formulas depend on the chosen orientation. Shape annotations are part of
the derivation, not decoration.


In [ ]:
import numpy as np
import torch

rng = np.random.default_rng(42)
B, I, O = 4, 3, 2
X = rng.normal(size=(B, I))       # [B,I]
W = rng.normal(size=(I, O))       # [I,O]
b = rng.normal(size=(O,))         # [O]
target = rng.normal(size=(B, O))  # [B,O]

Y = X @ W + b
error = Y - target
loss = np.mean(error**2)
dY = 2 * error / error.size
dX = dY @ W.T
dW = X.T @ dY
db = dY.sum(axis=0)
print("loss:", loss, "gradient shapes:", dX.shape, dW.shape, db.shape)


## 2. Broadcasting backward means summing expanded axes

If `[O]` bias broadcasts to `[B,O]`, each bias value influences all `B` rows, so its
gradient sums the batch axis. A general `unbroadcast` removes extra leading axes and
sums axes whose original dimension was 1.


In [ ]:
def unbroadcast(gradient: np.ndarray, original_shape: tuple[int, ...]) -> np.ndarray:
    while gradient.ndim > len(original_shape):
        gradient = gradient.sum(axis=0)
    for axis, size in enumerate(original_shape):
        if size == 1 and gradient.shape[axis] != 1:
            gradient = gradient.sum(axis=axis, keepdims=True)
    return gradient

upstream = np.ones((2, 5, 3))
assert unbroadcast(upstream, (3,)).shape == (3,)
assert np.all(unbroadcast(upstream, (3,)) == 10)
assert unbroadcast(upstream, (1, 5, 1)).shape == (1, 5, 1)
print("unbroadcast invariants passed")


## 3. Finite-difference gradient checking

For each parameter element, perturb by $\pm\epsilon$ and compare the central difference
with the analytical gradient. Use float64, a deterministic function, and relative error:

$$\frac{|g_a-g_n|}{\max(1,|g_a|,|g_n|)}.$$

Nondifferentiable points such as ReLU at zero require care because a numerical secant
need not match the framework's chosen subgradient.


In [ ]:
def finite_difference(array: np.ndarray, loss_fn, epsilon: float = 1e-6) -> np.ndarray:
    estimate = np.zeros_like(array)
    for index in np.ndindex(array.shape):
        old = array[index]
        array[index] = old + epsilon
        plus = loss_fn()
        array[index] = old - epsilon
        minus = loss_fn()
        array[index] = old
        estimate[index] = (plus - minus) / (2 * epsilon)
    return estimate

numeric_dW = finite_difference(W, lambda: np.mean((X @ W + b - target) ** 2))
relative_error = np.max(np.abs(dW - numeric_dW) / np.maximum(1, np.maximum(np.abs(dW), np.abs(numeric_dW))))
print("max relative error:", relative_error)
assert relative_error < 1e-8


## 4. PyTorch and `gradcheck`

PyTorch's `gradcheck` applies finite differences to a function and compares them with
autograd. It is designed for small float64 inputs. Passing a training run does not prove
every gradient is correct; direct local checks provide much stronger evidence.


In [ ]:
tX = torch.tensor(X, dtype=torch.float64, requires_grad=True)
tW = torch.tensor(W, dtype=torch.float64, requires_grad=True)
tb = torch.tensor(b, dtype=torch.float64, requires_grad=True)
ttarget = torch.tensor(target, dtype=torch.float64)
tloss = ((tX @ tW + tb - ttarget) ** 2).mean()
tloss.backward()

np.testing.assert_allclose(tX.grad.numpy(), dX, rtol=1e-10)
np.testing.assert_allclose(tW.grad.numpy(), dW, rtol=1e-10)
np.testing.assert_allclose(tb.grad.numpy(), db, rtol=1e-10)

def affine_loss(x, weight, bias):
    return ((x @ weight + bias - ttarget) ** 2).mean()

assert torch.autograd.gradcheck(affine_loss, (tX.detach().requires_grad_(),
                                              tW.detach().requires_grad_(),
                                              tb.detach().requires_grad_()))
print("manual, finite-difference, and PyTorch gradients agree")


## Failure modes to recognize

- **Missing mean factor:** gradients are scaled by batch or element count.
- **Broadcast gradient has output shape:** parameter updates cannot match the original parameter.
- **Float32 gradient check:** rounding hides or invents discrepancies.
- **Nondifferentiable test point:** valid subgradient conventions disagree with a symmetric finite difference.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Derive and check gradients for `tanh(X @ W + b)` followed by a sum.
2. Extend `unbroadcast` tests to scalar, leading-axis, and multiple-singleton cases.
3. Introduce one deliberate transpose bug and explain which gradient check catches it first.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can derive each tensor gradient by shape, reduce broadcasts correctly, and make three independent implementations agree.

**Next:** 25 — Modules and the training loop.
